In [1]:
import sqlite3
conn = sqlite3.connect('blockchain.db')

In [2]:
cur = conn.cursor()

In [3]:
cur.execute('''CREATE TABLE IF NOT EXISTS BLOCKS (
    hexString TEXT PRIMARY KEY,
    id INTEGER,
    view INTEGER,
    desc TEXT,
    img BLOB
)
''')

In [4]:
cur.execute('''CREATE TABLE IF NOT EXISTS VOTES (
    block_id TEXT,
    voter_id INTEGER,
    timestamp DATETIME,
    source_id INTEGER,
    FOREIGN KEY (block_id) REFERENCES BLOCKS(hexString),
    FOREIGN KEY (voter_id) REFERENCES PERSONS(id),
    FOREIGN KEY (source_id) REFERENCES SOURCES(id)
)
''')

In [5]:
cur.execute('''CREATE TABLE IF NOT EXISTS SOURCES (
    id INTEGER PRIMARY KEY,
    ip_addr TEXT,
    country_code TEXT
)
''')

In [6]:
cur.execute('''CREATE TABLE IF NOT EXISTS PERSONS (
    id INTEGER PRIMARY KEY,
    name TEXT,
    addr TEXT
)
''')

In [7]:
blocks_data = [
    ('0x1a2b3c4d', 1001, 101, 'Genesis Block', None),
    ('0x5e6f7a8b', 1002, 102, 'Second Block', None),
    ('0x9c0d1e2f', 1003, 103, 'Third Block', None)
]
cur.executemany('''
INSERT OR IGNORE INTO BLOCKS (hexString, id, view, desc, img)
VALUES (?, ?, ?, ?, ?)
''', blocks_data)

In [8]:
persons_data = [
    (1, 'John Doe', '123 Main St'),
    (2, 'Jane Smith', '456 Oak Ave'),
    (3, 'Bob Johnson', '789 Pine Rd')
]
cur.executemany('''
INSERT OR IGNORE INTO PERSONS (id, name, addr)
VALUES (?, ?, ?)
''', persons_data)

In [9]:
sources_data = [
    (1, '192.168.1.100', 'US'),
    (2, '10.0.0.50', 'UK'),
    (3, '172.16.0.25', 'DE'),
    (4, '192.168.1.200', 'FR')
]
cur.executemany('''
INSERT OR IGNORE INTO SOURCES (id, ip_addr, country_code)
VALUES (?, ?, ?)
''', sources_data)

In [10]:
votes_data = [
    ('0x1a2b3c4d', 1, '2024-01-15 10:30:00', 1),
    ('0x1a2b3c4d', 2, '2024-01-15 10:31:00', 2),
    ('0x5e6f7a8b', 1, '2024-01-15 10:32:00', 3),
    ('0x5e6f7a8b', 3, '2024-01-15 10:33:00', 1),
    ('0x9c0d1e2f', 2, '2024-01-15 10:34:00', 4),
    ('0x9c0d1e2f', 3, '2024-01-15 10:35:00', 2)
]
cur.executemany('''
INSERT INTO VOTES (block_id, voter_id, timestamp, source_id)
VALUES (?, ?, ?, ?)
''', votes_data)

In [11]:
conn.commit()

In [12]:
print("Таблиця BLOCKS:")
cur.execute("SELECT * FROM BLOCKS")
for row in cur.fetchall():
    print(f"hexString: {row[0]}, id: {row[1]}, view: {row[2]}, desc: {row[3]}")

Таблиця BLOCKS:
hexString: 0x1a2b3c4d, id: 1001, view: 101, desc: Genesis Block
hexString: 0x5e6f7a8b, id: 1002, view: 102, desc: Second Block
hexString: 0x9c0d1e2f, id: 1003, view: 103, desc: Third Block


In [13]:
print("Таблиця PERSONS:")
cur.execute("SELECT * FROM PERSONS")
for row in cur.fetchall():
    print(f"id: {row[0]}, name: {row[1]}, addr: {row[2]}")

Таблиця PERSONS:
id: 1, name: John Doe, addr: 123 Main St
id: 2, name: Jane Smith, addr: 456 Oak Ave
id: 3, name: Bob Johnson, addr: 789 Pine Rd


In [14]:
print("Таблиця SOURCES:")
cur.execute("SELECT * FROM SOURCES")
for row in cur.fetchall():
    print(f"id: {row[0]}, ip_addr: {row[1]}, country_code: {row[2]}")

Таблиця SOURCES:
id: 1, ip_addr: 192.168.1.100, country_code: US
id: 2, ip_addr: 10.0.0.50, country_code: UK
id: 3, ip_addr: 172.16.0.25, country_code: DE
id: 4, ip_addr: 192.168.1.200, country_code: FR


In [15]:
print("Таблиця VOTES:")
cur.execute("""SELECT v.block_id, v.voter_id, v.timestamp, v.source_id, p.name, s.ip_addr, b.desc
    FROM VOTES v
    LEFT JOIN PERSONS p ON v.voter_id = p.id
    LEFT JOIN SOURCES s ON v.source_id = s.id
    LEFT JOIN BLOCKS b ON v.block_id = b.hexString
""")
for row in cur.fetchall():
    print(f"block: {row[0]} ({row[6]}), voter: {row[1]} ({row[4]}), time: {row[2]}, source: {row[3]} ({row[5]})")


Таблиця VOTES:
block: 0x1a2b3c4d (Genesis Block), voter: 1 (John Doe), time: 2024-01-15 10:30:00, source: 1 (192.168.1.100)
block: 0x1a2b3c4d (Genesis Block), voter: 2 (Jane Smith), time: 2024-01-15 10:31:00, source: 2 (10.0.0.50)
block: 0x5e6f7a8b (Second Block), voter: 1 (John Doe), time: 2024-01-15 10:32:00, source: 3 (172.16.0.25)
block: 0x5e6f7a8b (Second Block), voter: 3 (Bob Johnson), time: 2024-01-15 10:33:00, source: 1 (192.168.1.100)
block: 0x9c0d1e2f (Third Block), voter: 2 (Jane Smith), time: 2024-01-15 10:34:00, source: 4 (192.168.1.200)
block: 0x9c0d1e2f (Third Block), voter: 3 (Bob Johnson), time: 2024-01-15 10:35:00, source: 2 (10.0.0.50)
block: 0x1a2b3c4d (Genesis Block), voter: 1 (John Doe), time: 2024-01-15 10:30:00, source: 1 (192.168.1.100)
block: 0x1a2b3c4d (Genesis Block), voter: 2 (Jane Smith), time: 2024-01-15 10:31:00, source: 2 (10.0.0.50)
block: 0x5e6f7a8b (Second Block), voter: 1 (John Doe), time: 2024-01-15 10:32:00, source: 3 (172.16.0.25)
block: 0x5e6f7a

In [16]:
cur.close()
conn.close()